In [ ]:
import scipy.io
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler  # 添加缺失的导入
from sklearn.ensemble import RandomForestRegressor  # 添加缺失的导入
from sklearn.multioutput import MultiOutputRegressor  # 添加缺失的导入

# 设置 Times New Roman 字体
plt.rcParams['font.sans-serif'] = ['Times New Roman']  # 使用 Times New Roman 字体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号 '-' 显示为方块的问题

# 1. 加载数据
print("正在加载数据...")
df_augmented = pd.read_excel('../data/CTGAN_grouped_data_multi.xlsx')
df_holdout = pd.read_excel('../data/holdout_data.xlsx')
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']
if len(df_holdout) != 500:
    raise ValueError('Expected untouched 500-record holdout exported separately from CTGAN training.')
X_augmented = df_augmented.drop(target_vars, axis=1)
y_augmented = df_augmented[target_vars]
X_test_orig = df_holdout.drop(target_vars, axis=1)
y_test_orig = df_holdout[target_vars]
scaler_X = StandardScaler()
X_augmented_scaled = scaler_X.fit_transform(X_augmented)
X_test_scaled = scaler_X.transform(X_test_orig)

model = MultiOutputRegressor(RandomForestRegressor(
    n_estimators=300,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    n_jobs=-1,
    random_state=42
))

# 使用增强数据训练
model.fit(X_augmented_scaled, y_augmented)
print("模型训练完成！")

# 7. 在测试集上进行预测
y_pred = model.predict(X_test_scaled)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

save_dir = './figures'
os.makedirs(save_dir, exist_ok=True)
for i, var_name in enumerate(target_vars):
    y_true = y_test_orig[var_name].to_numpy()
    y_hat = y_pred[:, i]
    residuals = y_true - y_hat
    plt.figure(figsize=(10, 5), dpi=300)
    plt.scatter(y_true, y_hat, s=12, alpha=0.7)
    lo = min(y_true.min(), y_hat.min())
    hi = max(y_true.max(), y_hat.max())
    plt.plot([lo, hi], [lo, hi], color='black', linestyle='--', label='Identity line')
    plt.xlabel('Observed holdout value')
    plt.ylabel('Predicted value')
    plt.title(f'Independent holdout predictions: {var_name}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{save_dir}/Holdout_prediction_{var_name}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'{var_name}: R?={r2_score(y_true, y_hat):.4f}, MAE={mean_absolute_error(y_true, y_hat):.4f}, RMSE={np.sqrt(mean_squared_error(y_true, y_hat)):.4f}, residual SD={np.std(residuals, ddof=1):.4f}')
print('Residual SD is descriptive only; no prediction/confidence interval is claimed.')